# MaxCut Benchmarking
## Cell 1 - Imports

In [1]:
import numpy as np
import pandas as pd
import time
import heapq
import math
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit_aer.aerprovider import AerSimulator
from scipy.optimize import minimize


from AdaptVQE import (
    build_reference_state,
    build_operator_pool,
    run_adapt_vqe,
    extract_result_metrics,
)


## Cell 1b - Noisy Backend Setup (FakeTorino)

In [2]:
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendEstimatorV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp as _SparsePauliOp

fake_torino = FakeTorino()
torino_simulator = AerSimulator.from_backend(fake_torino)

noisy_estimator = BackendEstimatorV2(backend=torino_simulator)

noisy_pm = generate_preset_pass_manager(optimization_level=1, backend=fake_torino, seed_transpiler=1234)

_isa_cache = {}

def to_isa_pub(circuit, observables, params, pm=noisy_pm):
    key = id(circuit)
    cached = _isa_cache.get(key)
    if cached is None or cached[0] is not circuit:
        isa_circuit = pm.run(circuit)
        isa_observables = [
            (obs if isinstance(obs, _SparsePauliOp) else _SparsePauliOp(obs)).apply_layout(isa_circuit.layout)
            for obs in observables
        ]
        cached = (circuit, isa_circuit, isa_observables)
        _isa_cache[key] = cached
    _, isa_circuit, isa_observables = cached
    return isa_circuit, isa_observables, params


def run_pub(circuit, observables, params, estimator, use_noise):
    if use_noise:
        isa_circuit, isa_observables, isa_params = to_isa_pub(circuit, observables, params)
        return estimator.run([(isa_circuit, isa_observables, isa_params)]).result()[0].data.evs
    return estimator.run([(circuit, observables, params)]).result()[0].data.evs

print(f"FakeTorino loaded: {fake_torino.num_qubits} qubits")
print(f"Basis gates: {sorted(fake_torino.operation_names)}")


FakeTorino loaded: 133 qubits
Basis gates: ['cz', 'delay', 'for_loop', 'id', 'if_else', 'measure', 'reset', 'rz', 'switch_case', 'sx', 'x']


## Cell 2 - Config

In [3]:
ideal_estimator = Estimator()

NOISY_SIM = True
estimator = noisy_estimator if NOISY_SIM else ideal_estimator

TRIALS = 3
COBYLA_MAXITER = 200
BETA1 = 0.9
BETA2 = 0.99
MB_ITERS = 3
SA_RUNS = 100
ADAPT_OPTIMIZER_MAXITER = 200
METHOD_ORDER = ["MomentumBuilder", "MomentumSA_Phased", "MomentumSA_Merged", "ADAPT-VQE"]
METHOD_COLORS = {
    "MomentumBuilder": "#E57373",
    "MomentumSA_Phased": "#9575CD",
    "MomentumSA_Merged": "#FFB74D",
    "ADAPT-VQE": "#4DB6AC",
}


## Cell 3 - Utility Functions

In [4]:
def cost_func(params, circuit, hamiltonian, estimator):
    cost = run_pub(circuit, hamiltonian, params, estimator, NOISY_SIM)
    return cost

def gradi(i, params, circuit, hamiltonian, estimator):
    delta = np.zeros(len(params))
    delta[i] = math.pi/2
    costp = cost_func(params+delta, circuit, hamiltonian, estimator)
    costm = cost_func(params-delta, circuit, hamiltonian, estimator)
    grad = (costp-costm)/2
    if isinstance(grad, np.ndarray):
        return grad[-1]
    return grad


## Cell 4 - MaxCut Hamiltonian Builder

In [5]:
def buildMaxCutHamiltonian(graph: nx.Graph) -> SparsePauliOp:
    """
    Build graph hamiltonian based on the following equation
    H = 1/2 ( Sum( w_ij (I - ZiZj) ) )

    Note: expectation value of this H on a computational basis state equals
    the cut value of that partition (higher = better cut). Since every VQE
    runner in this notebook MINIMIZES expectation value, we negate this
    Hamiltonian before benchmarking (see create_maxcut_hamiltonian below) so
    that minimizing corresponds to finding the MAX cut.
    """
    numQubits = graph.number_of_nodes()

    if not all(isinstance(n, int) and 0 <= n < numQubits for n in graph.nodes):
        nodeToQubit = {node: i for i, node in enumerate(graph.nodes)}
    else:
        nodeToQubit = {i: i for i in range(numQubits)}

    pauliTerms = []

    for u, v, data in graph.edges(data=True):
        weight = data.get('weight', 1.0)
        i = nodeToQubit[u]
        j = nodeToQubit[v]

        pauliTerms.append((Pauli('I' * numQubits), weight / 2))

        pauliStrings = ['I'] * numQubits
        pauliStrings[numQubits - 1 - i] = 'Z'
        pauliStrings[numQubits - 1 - j] = 'Z'
        pauliString = "".join(pauliStrings)
        pauliTerms.append((Pauli(pauliString), -weight/2))

    if not pauliTerms:
        return SparsePauliOp(Pauli('I' * numQubits), 0)

    paulis = [p for p, _ in pauliTerms]
    coeffs = [c for _, c in pauliTerms]
    H = SparsePauliOp(paulis, coeffs).simplify()
    return H


## Cell 4b - MaxCut Problems (12 graphs, 3-8 qubits)

In [6]:
def maxcut_problem1():
    # Square graph with heavy diagonal
    G = nx.Graph()
    G.add_edge(0, 1, weight=1.0)
    G.add_edge(1, 2, weight=1.0)
    G.add_edge(2, 3, weight=1.0)
    G.add_edge(3, 0, weight=1.0)
    G.add_edge(0, 2, weight=2.0)
    return G, 4.0

def maxcut_problem2():
    # Triangle with a heavy edge
    G = nx.Graph()
    G.add_edge(0, 1, weight=3.0)
    G.add_edge(0, 2, weight=1.0)
    G.add_edge(1, 2, weight=1.0)
    return G, 4.0

def maxcut_problem3():
    # Line graph with one heavy edge
    G = nx.Graph()
    G.add_edge(0, 1, weight=10.0)
    G.add_edge(1, 2, weight=1.0)
    return G, 11.0

def maxcut_problem4():
    # K5
    G = nx.complete_graph(5)
    return G, 6.0

def maxcut_problem5():
    # Line graph
    G = nx.path_graph(5)
    return G, 4.0

def maxcut_problem6():
    # Star graph
    G = nx.Graph()
    G.add_edges_from([(0, 1), (0, 2), (0, 3), (0, 4)], weight=1.0)
    return G, 4.0

def maxcut_problem7():
    # Two triangles bridged by a light edge
    G = nx.Graph()
    G.add_edge(0, 1, weight=5.0)
    G.add_edge(1, 2, weight=5.0)
    G.add_edge(0, 2, weight=5.0)
    G.add_edge(3, 4, weight=5.0)
    G.add_edge(4, 5, weight=5.0)
    G.add_edge(3, 5, weight=5.0)
    G.add_edge(2, 3, weight=1.0)
    return G, 21.0

def maxcut_problem8():
    # 4-node cycle with a very heavy anti-diagonal edge (1-3)
    G = nx.Graph()
    G.add_edge(0, 1, weight=1.0)
    G.add_edge(1, 2, weight=1.0)
    G.add_edge(2, 3, weight=1.0)
    G.add_edge(3, 0, weight=1.0)
    G.add_edge(1, 3, weight=10.0)
    return G, 12.0

def maxcut_problem9():
    # Complete bipartite K(3,3)
    G = nx.complete_bipartite_graph(3, 3)
    return G, 9.0

def maxcut_problem10():
    # Two disconnected edges
    G = nx.Graph()
    G.add_edge(0, 1, weight=5.0)
    G.add_edge(2, 3, weight=1.0)
    return G, 6.0

def maxcut_problem11():
    # 3-cube (8 nodes / 8 qubits)
    G = nx.hypercube_graph(3)
    return G, 12.0

def maxcut_problem12():
    # 5-cycle with one heavy chord
    G = nx.cycle_graph(5)
    G.add_edge(1, 4, weight=5.0)
    return G, 9.0


_MAXCUT_DEFS = {
    1: maxcut_problem1,   2: maxcut_problem2,   3: maxcut_problem3,
    4: maxcut_problem4,   5: maxcut_problem5,   6: maxcut_problem6,
    7: maxcut_problem7,   8: maxcut_problem8,   9: maxcut_problem9,
    10: maxcut_problem10, 11: maxcut_problem11, 12: maxcut_problem12,
}


def create_maxcut_hamiltonian(problem_id):
    """Return (H, expected_energy, n_qubits) for MaxCut problems 1-12.

    H is the NEGATED cut Hamiltonian (-1/2 * Sum w_ij (I - ZiZj)) so that
    minimizing <H> corresponds to finding the maximum cut. expected_energy
    is therefore -1 * (known max-cut value).
    """
    if problem_id not in _MAXCUT_DEFS:
        raise ValueError(f"Unknown maxcut problem {problem_id}")
    G, expectedCut = _MAXCUT_DEFS[problem_id]()
    H = -1.0 * buildMaxCutHamiltonian(G)
    return H, -expectedCut, G.number_of_nodes()


print("MaxCut Hamiltonian builder loaded (problems 1-12).")
print("H is negated so VQE minimization targets the MAX cut.")
print("Expected (negated) energies:")
for pid in range(1, 13):
    _, expected, nq = create_maxcut_hamiltonian(pid)
    print(f"  Problem {pid:2d}: qubits={nq} | expected_energy={expected:6.2f}  (max-cut={-expected:.2f})")


MaxCut Hamiltonian builder loaded (problems 1-12).
H is negated so VQE minimization targets the MAX cut.
Expected (negated) energies:
  Problem  1: qubits=4 | expected_energy= -4.00  (max-cut=4.00)
  Problem  2: qubits=3 | expected_energy= -4.00  (max-cut=4.00)
  Problem  3: qubits=3 | expected_energy=-11.00  (max-cut=11.00)
  Problem  4: qubits=5 | expected_energy= -6.00  (max-cut=6.00)
  Problem  5: qubits=5 | expected_energy= -4.00  (max-cut=4.00)
  Problem  6: qubits=5 | expected_energy= -4.00  (max-cut=4.00)
  Problem  7: qubits=6 | expected_energy=-21.00  (max-cut=21.00)
  Problem  8: qubits=4 | expected_energy=-12.00  (max-cut=12.00)
  Problem  9: qubits=6 | expected_energy= -9.00  (max-cut=9.00)
  Problem 10: qubits=4 | expected_energy= -6.00  (max-cut=6.00)
  Problem 11: qubits=8 | expected_energy=-12.00  (max-cut=12.00)
  Problem 12: qubits=5 | expected_energy= -9.00  (max-cut=9.00)


## Cell 5 - Momentum Functions

In [7]:
def momen_layer(it, n, momentum, radius=1, keep=2):
    lay = QuantumCircuit(n)
    params, inds = [], []
    actual_keep = min(keep, len(momentum))

    for i in range(actual_keep):
        if len(momentum) == 0:
            break
        m_val, ind = heapq.heappop(momentum)
        angle = Parameter(f"it{it}_q{i}")
        params.append(1)
        inds.append(ind)
        lay.rx(angle, ind)
        for r in range(1, radius+1):
            if ind + r < n:
                lay.cx(ind, ind+r)
            if ind - r >= 0:
                lay.cx(ind, ind-r)
    return lay, params, inds

def MomentumBuilder(params, inds, ansatz, circuit, hamiltonian, estimator, beta1, beta2, iters=2):
    n = circuit.num_qubits
    M = np.zeros(len(params))
    currCirc = QuantumCircuit(n).compose(ansatz)
    observables = [*hamiltonian.paulis, hamiltonian]

    for it in range(iters):
        accumulator = []
        for i in range(len(params)):
            grad_i = abs(gradi(i, params, currCirc, observables, estimator))
            M[i] = beta1 * M[i] + (1 - beta1) * grad_i
            heapq.heappush(accumulator, (-M[i], inds[i]))

        keep = max(2, min(n // 2, len(accumulator)))
        mLayer, nparams, ninds = momen_layer(it, n, accumulator, keep=keep)
        params = params + nparams
        inds = inds + ninds
        M = np.concatenate((M, np.zeros(len(nparams))))
        ansatz = ansatz.compose(mLayer)
        currCirc = circuit.compose(ansatz)

    return circuit.compose(ansatz)

def simulated_annealing(optimization_runs, initial_params, circuit, simulator, observables, estimator):
    def evaluate(params):
        return run_pub(circuit, observables, params, estimator, NOISY_SIM)[-1]

    current = np.array(initial_params)
    current_E = evaluate(current)
    best, best_E = current.copy(), current_E

    for i in range(optimization_runs):
        T = 2.0 * (0.01 / 2.0) ** (i / optimization_runs)
        proposal = current + np.random.normal(0, 0.3 * T, len(current))
        proposal_E = evaluate(proposal)

        if proposal_E < current_E or np.random.random() < np.exp(-(proposal_E - current_E) / T):
            current, current_E = proposal, proposal_E
            if current_E < best_E:
                best, best_E = current.copy(), current_E
    return best


## Cell 6 - Momentum Methods

In [8]:
def momentum_sa_phased(params, inds, ansatz, circuit, H, estimator, beta1, beta2, iters, opt_runs):
    opt_ansatz = MomentumBuilder(params, inds, ansatz, circuit, H, estimator, beta1, beta2, iters)
    init_params = np.random.uniform(-np.pi, np.pi, len(opt_ansatz.parameters))
    obs = [*H.paulis, H]
    final_params = simulated_annealing(opt_runs, init_params, opt_ansatz, None, obs, estimator)
    return opt_ansatz, final_params, cost_func(final_params, opt_ansatz, obs, estimator)[-1]

def momentum_sa_merged(params, inds, ansatz, circuit, H, estimator, beta1, beta2, iters, opt_runs):
    n = circuit.num_qubits
    obs = [*H.paulis, H]
    params = list(np.random.uniform(-np.pi, np.pi, len(params)))
    inds = list(inds)
    M = np.zeros(len(params))
    currCirc = QuantumCircuit(n).compose(ansatz)

    for it in range(iters):
        acc = []
        for i in range(len(params)):
            grad_i = abs(gradi(i, np.array(params), currCirc, obs, estimator))
            M[i] = beta1 * M[i] + (1 - beta1) * grad_i
            heapq.heappush(acc, (-M[i], inds[i]))

        keep = max(2, min(n // 2, len(acc)))
        mLayer, nparams, ninds = momen_layer(it, n, acc, keep=keep)
        params += nparams
        inds += ninds
        M = np.concatenate((M, np.zeros(len(nparams))))
        ansatz = ansatz.compose(mLayer)
        currCirc = circuit.compose(ansatz)
        params = list(simulated_annealing(opt_runs, np.array(params), currCirc, None, obs, estimator))

    circuit = circuit.compose(ansatz)
    return circuit, np.array(params), cost_func(np.array(params), circuit, obs, estimator)[-1]


## Cell 7 - Test Problems

In [9]:
selected = []

print("Creating MaxCut test problems...")
for prob_id in range(1, 13):
    try:
        H, expected, n_qubits = create_maxcut_hamiltonian(prob_id)
        selected.append((prob_id, H, expected, n_qubits))
        print(f"\u2713 Problem {prob_id:2d} (MaxCut) | Qubits: {n_qubits} | Expected: {expected:6.2f}")
    except Exception as e:
        print(f"\u2717 Problem {prob_id:2d} FAILED: {e}")
        break

print("\n" + "=" * 80)
print(f"SUCCESSFULLY LOADED {len(selected)} MAXCUT PROBLEMS")
print("=" * 80)
for prob_id, H, expected, n_qubits in selected:
    print(f"Problem {prob_id:2d} (MaxCut) | Qubits: {n_qubits} | Expected: {expected:6.2f} | Terms: {len(H)}")
print("=" * 80)


Creating MaxCut test problems...
✓ Problem  1 (MaxCut) | Qubits: 4 | Expected:  -4.00
✓ Problem  2 (MaxCut) | Qubits: 3 | Expected:  -4.00
✓ Problem  3 (MaxCut) | Qubits: 3 | Expected: -11.00
✓ Problem  4 (MaxCut) | Qubits: 5 | Expected:  -6.00
✓ Problem  5 (MaxCut) | Qubits: 5 | Expected:  -4.00
✓ Problem  6 (MaxCut) | Qubits: 5 | Expected:  -4.00
✓ Problem  7 (MaxCut) | Qubits: 6 | Expected: -21.00
✓ Problem  8 (MaxCut) | Qubits: 4 | Expected: -12.00
✓ Problem  9 (MaxCut) | Qubits: 6 | Expected:  -9.00
✓ Problem 10 (MaxCut) | Qubits: 4 | Expected:  -6.00
✓ Problem 11 (MaxCut) | Qubits: 8 | Expected: -12.00
✓ Problem 12 (MaxCut) | Qubits: 5 | Expected:  -9.00

SUCCESSFULLY LOADED 12 MAXCUT PROBLEMS
Problem  1 (MaxCut) | Qubits: 4 | Expected:  -4.00 | Terms: 6
Problem  2 (MaxCut) | Qubits: 3 | Expected:  -4.00 | Terms: 4
Problem  3 (MaxCut) | Qubits: 3 | Expected: -11.00 | Terms: 3
Problem  4 (MaxCut) | Qubits: 5 | Expected:  -6.00 | Terms: 11
Problem  5 (MaxCut) | Qubits: 5 | Expected

## Cell 8 - Runner Functions

In [10]:
def evaluate_energy(circuit, H, params):
    obs = [*H.paulis, H]
    return float(run_pub(circuit, obs, params, estimator, NOISY_SIM)[-1])


def create_initial_ansatz(n):
    qc = QuantumCircuit(n)
    theta = ParameterVector("theta", n)
    for i in range(n):
        qc.rx(theta[i], i)
    return qc, list(np.random.uniform(-0.5, 0.5, n)), list(range(n))


def run_momentum_builder(H):
    n = H.num_qubits
    ansatz, params, inds = create_initial_ansatz(n)
    t0 = time.perf_counter()
    opt_circuit = MomentumBuilder(params, inds, ansatz, QuantumCircuit(n), H, estimator, BETA1, BETA2, MB_ITERS)
    final_params = np.ones(len(opt_circuit.parameters))
    energy = evaluate_energy(opt_circuit, H, final_params)
    return energy, time.perf_counter() - t0, len(opt_circuit.parameters)


def run_momentum_sa_phased(H):
    n = H.num_qubits
    ansatz, params, inds = create_initial_ansatz(n)
    t0 = time.perf_counter()
    _, _, energy = momentum_sa_phased(params, inds, ansatz, QuantumCircuit(n), H, estimator, BETA1, BETA2, MB_ITERS, SA_RUNS)
    return energy, time.perf_counter() - t0, len(params) + MB_ITERS * max(2, n // 2)


def run_momentum_sa_merged(H):
    n = H.num_qubits
    ansatz, params, inds = create_initial_ansatz(n)
    t0 = time.perf_counter()
    _, _, energy = momentum_sa_merged(params, inds, ansatz, QuantumCircuit(n), H, estimator, BETA1, BETA2, MB_ITERS, SA_RUNS)
    return energy, time.perf_counter() - t0, len(params) + MB_ITERS * max(2, n // 2)


def run_adapt_algorithm(H):
    t0 = time.perf_counter()
    adapt_kwargs = dict(
        hamiltonian=H,
        initial_state=build_reference_state(H.num_qubits, angle=1.0),
        operator_pool=build_operator_pool(H.num_qubits),
        max_iterations=max(12, H.num_qubits),
        gradient_threshold=1e-6,
        eigenvalue_threshold=1e-8,
        optimizer_maxiter=ADAPT_OPTIMIZER_MAXITER,
    )
    try:
        result = run_adapt_vqe(estimator=estimator, **adapt_kwargs)
    except TypeError:
        result = run_adapt_vqe(**adapt_kwargs)
    elapsed = time.perf_counter() - t0
    metrics = extract_result_metrics(result)
    energy = float(np.real(metrics["energy"]))
    params = metrics.get("num_parameters")
    params = int(params) if params is not None else 0
    return energy, elapsed, params, metrics.get("num_iterations"), metrics.get("termination_criterion")


## Cell 9 - Benchmark

In [ ]:
results = []
print("=" * 80)
print(f"BENCHMARKING {len(selected)} MAXCUT PROBLEMS \u00d7 {TRIALS} TRIALS \u00d7 4 METHODS")
print("=" * 80)

for prob_id, H, expected, n_qubits in selected:
    print(f"\nProblem {prob_id:2d} | Qubits: {n_qubits} | Expected: {expected:.2f}")
    print("-" * 80)

    for trial in range(TRIALS):
        np.random.seed(1000 + prob_id * 10 + trial)
        print(f"  Trial {trial + 1}")

        e, t, p = run_momentum_builder(H)
        print(f"    MomentumBuilder   : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s")
        results.append((prob_id, trial + 1, "MomentumBuilder", e, t, p, expected, np.nan, None))

        e, t, p = run_momentum_sa_phased(H)
        print(f"    MomentumSA_Phased : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s")
        results.append((prob_id, trial + 1, "MomentumSA_Phased", e, t, p, expected, np.nan, None))

        e, t, p = run_momentum_sa_merged(H)
        print(f"    MomentumSA_Merged : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s")
        results.append((prob_id, trial + 1, "MomentumSA_Merged", e, t, p, expected, np.nan, None))

        e, t, p, adapt_iters, term = run_adapt_algorithm(H)
        print(f"    ADAPT-VQE         : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s | iters={adapt_iters}")
        results.append((prob_id, trial + 1, "ADAPT-VQE", e, t, p, expected, adapt_iters, term))

df = pd.DataFrame(
    results,
    columns=[
        "problem",
        "trial",
        "algorithm",
        "energy",
        "time",
        "params",
        "expected",
        "adapt_iterations",
        "termination_criterion",
    ],
)
df["gap"] = df["energy"] - df["expected"]
df["abs_gap"] = abs(df["gap"])
df["algorithm"] = pd.Categorical(df["algorithm"], categories=METHOD_ORDER, ordered=True)
df = df.sort_values(["problem", "trial", "algorithm"]).reset_index(drop=True)

print("\n" + "=" * 80)
print("COMPLETE")
print("=" * 80)


BENCHMARKING 12 MAXCUT PROBLEMS × 3 TRIALS × 4 METHODS

Problem  1 | Qubits: 4 | Expected: -4.00
--------------------------------------------------------------------------------
  Trial 1


## Cell 10 - Results

In [ ]:
display(df)
print("\nSummary by Algorithm:")
summary = df.groupby("algorithm", observed=False).agg({
    "abs_gap": ["mean", "std", "min", "max"],
    "time": "mean",
    "params": "mean",
    "adapt_iterations": "mean",
}).reindex(METHOD_ORDER).dropna(how="all")
display(summary)
print("\nBest per Problem:")
best = df.loc[df.groupby("problem")["energy"].idxmin()][["problem", "algorithm", "energy", "expected", "gap"]]
display(best.sort_values("problem"))


In [ ]:
# Group by algorithm and compute summary statistics
summary = df.groupby('algorithm', observed=False).agg({
    'energy': ['mean', 'std', 'min', 'max'],
    'time': ['mean', 'std'],
    'params': ['mean', 'std'],
    'adapt_iterations': ['mean', 'std'],
})

print("\n" + "=" * 80)
print("SUMMARY STATISTICS BY ALGORITHM")
print("=" * 80)
display(summary.reindex(METHOD_ORDER).dropna(how='all'))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
available = [alg for alg in METHOD_ORDER if alg in set(df['algorithm'].astype(str))]

energy_data = [df[df['algorithm'] == alg]['energy'].dropna().values for alg in available]
bplot1 = axes[0, 0].boxplot(energy_data, tick_labels=available, patch_artist=True)
for patch, alg in zip(bplot1['boxes'], available):
    patch.set_facecolor(METHOD_COLORS.get(alg, '#888888'))
    patch.set_alpha(0.7)
axes[0, 0].set_title('Final Energy Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Energy (negated cut value)', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)


time_data = [df[df['algorithm'] == alg]['time'].dropna().values for alg in available]
bplot2 = axes[0, 1].boxplot(time_data, tick_labels=available, patch_artist=True)
for patch, alg in zip(bplot2['boxes'], available):
    patch.set_facecolor(METHOD_COLORS.get(alg, '#888888'))
    patch.set_alpha(0.7)
axes[0, 1].set_title('Optimization Time Comparison', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Time (seconds)', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

param_means = df.groupby('algorithm', observed=False)['params'].mean().reindex(available).dropna()
bars = axes[1, 0].bar(range(len(param_means)), param_means.values)
for bar, alg in zip(bars, param_means.index):
    bar.set_color(METHOD_COLORS.get(alg, '#888888'))
    bar.set_alpha(0.7)
axes[1, 0].set_title('Number of Parameters', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Parameters', fontsize=12)
axes[1, 0].set_xticks(range(len(param_means)))
axes[1, 0].set_xticklabels(param_means.index, rotation=15, ha='right')
axes[1, 0].grid(True, alpha=0.3, axis='y')

for alg in available:
    subset = df[df['algorithm'] == alg]
    axes[1, 1].scatter(
        subset['time'],
        subset['energy'],
        label=alg,
        alpha=0.6,
        s=50,
        color=METHOD_COLORS.get(alg, '#888888'),
    )
axes[1, 1].set_title('Energy vs Time Trade-off', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Time (seconds)', fontsize=12)
axes[1, 1].set_ylabel('Energy', fontsize=12)
axes[1, 1].legend(loc='upper right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
means = df.groupby("algorithm", observed=False).mean(numeric_only=True).reindex(METHOD_ORDER).dropna(how="all")

print("=" * 80)
print("PERFORMANCE SUMMARY - MAXCUT BENCHMARK")
print("=" * 80)

if len(means) > 0:
    print("\nMean metrics by algorithm:")
    display(means[["energy", "abs_gap", "time", "params", "adapt_iterations"]])

    if "ADAPT-VQE" in means.index:
        for baseline in ["MomentumBuilder", "MomentumSA_Phased", "MomentumSA_Merged"]:
            if baseline not in means.index:
                continue

            baseline_abs_gap = means.loc[baseline, "abs_gap"]
            adapt_abs_gap = means.loc["ADAPT-VQE", "abs_gap"]
            gap_reduction = np.nan
            if baseline_abs_gap != 0:
                gap_reduction = ((baseline_abs_gap - adapt_abs_gap) / baseline_abs_gap) * 100

            energy_improvement = (
                (means.loc[baseline, "energy"] - means.loc["ADAPT-VQE", "energy"])
                / abs(means.loc[baseline, "energy"])
            ) * 100
            speed_ratio = means.loc[baseline, "time"] / means.loc["ADAPT-VQE", "time"]

            print("\n" + "=" * 80)
            print(f"ADAPT-VQE vs {baseline}")
            print("=" * 80)
            print(f"Energy Improvement:  {energy_improvement:+7.2f}%")
            print(f"Gap Reduction:       {gap_reduction:+7.2f}%")
            print(f"Speed Ratio:         {speed_ratio:7.2f}x")

    print("\n" + "=" * 80)
    print("WIN COUNT (Best Energy Per Problem)")
    print("=" * 80)
    best = df.loc[df.groupby('problem')['energy'].idxmin()]
    wins = best['algorithm'].value_counts().reindex(METHOD_ORDER).fillna(0).astype(int)
    for alg, count in wins.items():
        print(f"{alg:20s}: {count:2d} / {len(selected)} problems")
else:
    print("Missing algorithms in results.")
